# Xuất MobileNet thắng cuộc sang ONNX

Notebook này không train. Nó xác minh quyết định đã đóng băng từ validation, xuất checkpoint MobileNet sang ONNX fixed batch 1, kiểm tra ONNX, chạy parity PyTorch/ONNX trên ít nhất 20 ảnh test cân bằng và tạo bundle để inference trên laptop.

In [ ]:
REPO_URL = "https://github.com/DangHung1110/IoT-CK.git"
GIT_REF = "feat/model-training-pipeline"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/parcel_sorter"  # @param {type:"string"}
MOBILENET_RUN_ID = "20261103t153037z-mobilenet_v3_small-v0-s42-7c300430"  # @param {type:"string"}
EVALUATION_ID = "20261104t015526z-binary-v0-7c300430"  # @param {type:"string"}
ONNX_OPSET = 18
PARITY_SAMPLES_PER_CLASS = 10
PARITY_TOLERANCE = 1e-4

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import re
import subprocess
from pathlib import Path

repo_dir = Path("/content/IoT-CK")
if not (repo_dir / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(repo_dir)],
        check=True,
    )
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", "--prune"], check=True)
target_ref = GIT_REF if re.fullmatch(r"[0-9a-f]{40}", GIT_REF) else f"origin/{GIT_REF}"
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", target_ref], check=True)

In [ ]:
import importlib
import sys

if not ((3, 11) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError(f"Unsupported Colab Python: {sys.version.split()[0]}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir), "--no-deps"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "onnx==1.23.2", "onnxruntime==1.31.1"],
    check=True,
)
src_dir = str(repo_dir / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
for module_name in tuple(sys.modules):
    if module_name == "parcel_sorter" or module_name.startswith("parcel_sorter."):
        del sys.modules[module_name]
importlib.invalidate_caches()
parcel_sorter = importlib.import_module("parcel_sorter")
package_path = Path(parcel_sorter.__file__).resolve()
if not package_path.is_relative_to(repo_dir.resolve()):
    raise RuntimeError(f"parcel_sorter loaded from unexpected path: {package_path}")
print(f"Loaded parcel_sorter from {package_path}")

In [ ]:
import json
import shutil

from parcel_sorter.colab import validate_colab_drive_root
from parcel_sorter.data import BinaryParcelClass, DatasetSplit, sha256_file
from parcel_sorter.ml.binary_evaluation import BinaryThresholds
from parcel_sorter.ml.colab_data import IMAGE_SUFFIXES, stage_verified_dataset
from parcel_sorter.ml.colab_evaluation import resolve_best_checkpoint
from parcel_sorter.ml.contracts import ModelCandidate
from parcel_sorter.ml.run_metadata import load_training_run

drive_root = validate_colab_drive_root(DRIVE_ROOT)
run_dir = drive_root / "models" / "runs" / MOBILENET_RUN_ID
evaluation_dir = drive_root / "models" / "evaluations" / EVALUATION_ID
manifest = load_training_run(run_dir)
if manifest.config.candidate is not ModelCandidate.MOBILENET_V3_SMALL:
    raise RuntimeError("Selected run is not MobileNetV3 Small")
selection_path = evaluation_dir / "selection_frozen.json"
comparison_path = evaluation_dir / "comparison.json"
selection = json.loads(selection_path.read_text(encoding="utf-8"))
comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
if selection["selection_basis"] != "validation_only":
    raise RuntimeError("Model selection was not validation-only")
if selection["winner"] != ModelCandidate.MOBILENET_V3_SMALL.value:
    raise RuntimeError("Frozen winner is not MobileNetV3 Small")
if comparison["winner"] != selection["winner"] or not comparison["test_did_not_change_winner"]:
    raise RuntimeError("Evaluation comparison does not preserve the frozen winner")
checkpoint = resolve_best_checkpoint(run_dir, ModelCandidate.MOBILENET_V3_SMALL)
checkpoint_sha256 = sha256_file(checkpoint)
if checkpoint_sha256 != selection["winner_checkpoint_sha256"]:
    raise RuntimeError("Checkpoint hash does not match frozen selection")
thresholds = BinaryThresholds.model_validate(selection["winner_thresholds"])
workspace = Path("/content/parcel_sorter_export_data")
if workspace.exists():
    shutil.rmtree(workspace)
prepared = stage_verified_dataset(
    archive_source=manifest.dataset.archive_source,
    manifest_source=manifest.dataset.manifest_source,
    expected_archive_sha256=manifest.dataset.archive_sha256,
    expected_manifest_sha256=manifest.dataset.manifest_sha256,
    drive_root=drive_root,
    local_workspace=workspace,
    class_order=manifest.dataset.class_order,
)
print(f"Verified winner checkpoint: {checkpoint_sha256}")

In [ ]:
import onnx
import torch
import torchvision

config = manifest.config
model = torchvision.models.mobilenet_v3_small(weights=None)
in_features = model.classifier[-1].in_features
model.classifier[-1] = torch.nn.Linear(in_features, len(config.classes))
payload = torch.load(checkpoint, map_location="cpu", weights_only=False)
if payload.get("config") != config.model_dump(mode="json"):
    raise RuntimeError("Checkpoint config does not match run manifest")
if tuple(payload.get("class_order", ())) != tuple(item.value for item in config.classes):
    raise RuntimeError("Checkpoint class order does not match run manifest")
model.load_state_dict(payload["model_state"])
model.eval()
local_bundle = Path("/content/binary_v0_onnx_bundle")
if local_bundle.exists():
    shutil.rmtree(local_bundle)
local_bundle.mkdir()
onnx_path = local_bundle / "model.onnx"
dummy = torch.zeros(1, 3, config.preprocess.input_size, config.preprocess.input_size)
with torch.no_grad():
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        input_names=["images"],
        output_names=["logits"],
        opset_version=ONNX_OPSET,
        dynamo=False,
    )
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print(f"Valid ONNX: {onnx_path} ({onnx_path.stat().st_size / 1024 / 1024:.2f} MiB)")

In [ ]:
import onnxruntime as ort
from PIL import Image

from parcel_sorter.ml.binary_artifacts import (
    BinaryOnnxBundleManifest,
    save_binary_bundle_manifest,
    verify_binary_onnx_parity,
)

test_root = prepared.splits[DatasetSplit.TEST].path
sample_paths = []
for parcel_class in config.classes:
    candidates = sorted(
        path
        for path in (test_root / parcel_class.value).rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    if len(candidates) < PARITY_SAMPLES_PER_CLASS:
        raise RuntimeError(f"Not enough test images for parity: {parcel_class.value}")
    sample_paths.extend(candidates[:PARITY_SAMPLES_PER_CLASS])
transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize(
            (config.preprocess.input_size, config.preprocess.input_size),
            interpolation=torchvision.transforms.InterpolationMode.BILINEAR,
            antialias=True,
        ),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=config.preprocess.mean, std=config.preprocess.std),
    ]
)
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
torch_rows = []
onnx_rows = []
with torch.no_grad():
    for image_path in sample_paths:
        with Image.open(image_path) as image:
            tensor = transform(image.convert("RGB")).unsqueeze(0)
        torch_rows.append(torch.softmax(model(tensor), dim=1)[0].tolist())
        logits = torch.from_numpy(session.run(["logits"], {"images": tensor.numpy()})[0])
        onnx_rows.append(torch.softmax(logits, dim=1)[0].tolist())
classes = tuple(BinaryParcelClass(item.value) for item in config.classes)
parity = verify_binary_onnx_parity(
    torch_rows,
    onnx_rows,
    classes=classes,
    thresholds=thresholds,
    absolute_tolerance=PARITY_TOLERANCE,
)
if not parity.numerical_parity or not parity.decision_parity:
    raise RuntimeError(f"ONNX parity failed: {parity.model_dump()}")
bundle_manifest = BinaryOnnxBundleManifest(
    candidate=ModelCandidate.MOBILENET_V3_SMALL,
    run_id=manifest.run_id,
    evaluation_id=EVALUATION_ID,
    training_git_commit=manifest.git_commit,
    dataset_archive_sha256=manifest.dataset.archive_sha256,
    dataset_manifest_sha256=manifest.dataset.manifest_sha256,
    source_checkpoint_sha256=checkpoint_sha256,
    frozen_selection_sha256=sha256_file(selection_path),
    onnx_sha256=sha256_file(onnx_path),
    onnx_opset=ONNX_OPSET,
    input_name="images",
    output_name="logits",
    classes=classes,
    preprocess=config.preprocess,
    thresholds=thresholds,
    parity=parity,
)
save_binary_bundle_manifest(local_bundle / "manifest.json", bundle_manifest)
bundle_id = f"{EVALUATION_ID}-mobilenet-v3-small-onnx"
bundle_dir = drive_root / "models" / "exports" / bundle_id
if bundle_dir.exists():
    raise FileExistsError(f"Refusing to overwrite existing bundle: {bundle_dir}")
bundle_dir.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(local_bundle, bundle_dir)
archive_path = Path(shutil.make_archive(str(bundle_dir), "zip", root_dir=bundle_dir))
print(parity.model_dump_json(indent=2))
print(f"Bundle: {bundle_dir}")
print(f"Download this ZIP for laptop inference: {archive_path}")